In [ ]:
!pip install "numpy<2" --force-reinstall

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 777.1 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 35.5 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
tobler 0.14.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
shap 0.52.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
pytensor 2.38.3 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 wh

In [ ]:
!pip install xlrd openpyxl

In [ ]:
!pip install elasticsearch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 993.6/993.6 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.2/66.2 kB 5.7 MB/s eta 0:00:00


In [ ]:
!pip install opencv-python ultralytics numpy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 666.6 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 59.1 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.4.6 which is incompatible.


In [ ]:
import cv2
import numpy as np
from ultralytics import YOLO
from collections import defaultdict
import pandas as pd

In [ ]:
""" This code block gets the dimensions of the video file uploaded from NYS-DOT"""

video_stream_url = "https://s9.nysdot.skyvdn.com:443/rtplive/R1_059/playlist.m3u8"

# 1. Capture the live video stream
cap = cv2.VideoCapture(video_stream_url)

# 2. Check if the stream opened successfully
if not cap.isOpened():
    print("Error: Could not open the video stream. Check the URL or your network connection.")
else:
    # 3. Retrieve the pixel measurements using OpenCV properties
    width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))   # Pixel width (X-axis)
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))  # Pixel height (Y-axis)
    fps    = cap.get(cv2.CAP_PROP_FPS)                # Frames per second

    # 4. Calculate your specific 65% runner line threshold based on the height
    runner_line_y = int(height * 0.65)

    # Print the measurements
    print("--- Video Stream Dimensions ---")
    print(f"Width:  {width} pixels")
    print(f"Height: {height} pixels")
    print(f"Frames Per Second (FPS): {fps}")
    print(f"Your Runner Line (65% down): Y = {runner_line_y}")

    # 5. Optional: Read a single frame just to confirm it works
    ret, frame = cap.read()
    if ret:
        print(f"Successfully read the first frame matrix shape: {frame.shape}")

    # 6. Always release the stream when finished
    cap.release()

--- Video Stream Dimensions ---
Width:  320 pixels
Height: 180 pixels
Frames Per Second (FPS): 15.0
Your Runner Line (65% down): Y = 117
Successfully read the first frame matrix shape: (180, 320, 3)


In [ ]:
""" This code block reads a video coming in from the NYS department of transportation
and scripts attributes of the video into a dataset that can be then uploaded to Elastic"""

import csv
import pandas as pd
from datetime import datetime
from elasticsearch import Elasticsearch
from elasticsearch.helpers import bulk

# 1. Provide your Elastic Cloud ID and credentials
# You can use basic_auth (username/password) or api_key here
CLOUD_ID = "HackathonTrafficMonitor:dXMtY2VudHJhbDEuZ2NwLmNsb3VkLmVzLmlvOjQ0MyQyYzRiMmJjMDg4ZDg0OWRjODlhZmFjMWFlNzRkMGNhZSRhOTk1ZmE0MWJiOTQ0MmMwYmNjYzE0ZDYwMmNhYzM1NA=="
PASSWORD = "xeStqhvi3rK5WyCrocMmr8TE"

es = Elasticsearch(
    cloud_id=CLOUD_ID,
    basic_auth=("elastic", PASSWORD)
)

# Set path to your Excel file (.xls)
excel_file_path = "traffic_1hour_dataset.xls"
index_name = "traffic-1hour-analysis"

print("Reading Excel file via Pandas...")

# 2. Use Pandas to read the Excel file cleanly
df = pd.read_csv("/content/traffic_1hour_dataset.xls")

# Normalize column names to uppercase and strip trailing spaces to prevent typos
df.columns = df.columns.str.strip().str.upper()

# Handle any empty cells by filling them with standard defaults
df = df.fillna(0)

# 3. Generator function translating the Pandas rows into Elastic JSON docs
def generate_elastic_docs():
    for _, row in df.iterrows():
        yield {
            "_index": index_name,
            "_source": {
                "timestamp": str(row.get("TIMESTAMP", "")),
                "OBJECT_ID": int(row.get("OBJECT_ID", 0)),
                "MOVING_AWAY": int(row.get("MOVING_AWAY", 0)),
                "MOVING_TOWARD": int(row.get("MOVING_TOWARD", 0)),
                "VELOCITY": float(row.get("VELOCITY", 0.0)),
                "ACCELERATION": float(row.get("ACCELERATION", 0.0)),
                "MAX_IMPACT_SCORE": float(row.get("MAX_IMPACT_SCORE", 0.0)),
                "COUNT_DOWN_RATE": float(row.get("COUNT_DOWN_RATE", 1.0))
            }
        }

# 4. Stream everything directly to your cloud instance
try:
    success, errors = bulk(es, generate_elastic_docs())
    print(f"🎉 Success! Uploaded {success} rows directly from Excel to Elastic Cloud.")
    es.indices.refresh(index=index_name)
except Exception as e:
    print(f"❌ Upload failed: {e}")

Reading Excel file via Pandas...
🎉 Success! Uploaded 216 rows directly from Excel to Elastic Cloud.


In [ ]:
""""This code block demos the back end of the block above. This shows each object as
it is identified as a moving car and labels it with wether it is moving toward, moving away
passed, or idle in the video frame"""


import cv2
from ultralytics import YOLO
from collections import defaultdict
import time

# 1. Initialize YOLOv8 Nano model (lightweight for fast tracking)
model = YOLO("yolov8n.pt")

# Live NYSDOT Stream URL (I-87 Segment)
video_source = "https://s9.nysdot.skyvdn.com:443/rtplive/R1_2109/playlist.m3u8"
cap = cv2.VideoCapture(video_source)

# Dictionary to maintain historical frame positions for tracking speeds
track_history = defaultdict(lambda: [])

# Configuration parameters for the tracking physics
FRAME_WINDOW = 8   # Number of past frames used to smooth out velocity vectors
ASSUMED_FPS = 30   # Stream frame rate baseline
target_y_line = None

print("🎬 Initializing Live NYSDOT Tracking Demo...")
print("👉 Click inside the popup video window and press 'q' to quit.")

while cap.isOpened():
    success, frame = cap.read()
    if not success:
        print("Video stream buffering or offline...")
        break

    # Dynamically extract frame size properties
    h_frame, w_frame, _ = frame.shape
    if target_y_line is None:
        # Establish the virtual runner position at 65% down the screen
        target_y_line = int(h_frame * 0.65)

    # Draw the static target line across the frame
    cv2.line(frame, (0, target_y_line), (w_frame, target_y_line), (0, 165, 255), 3)
    cv2.putText(frame, "RUNNER POSITION (TARGET LINE)", (20, target_y_line - 15),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 165, 255), 2)

    # 2. Track vehicles (Classes: 2=car, 3=motorcycle, 5=bus, 7=truck)
    results = model.track(frame, persist=True, classes=[2, 3, 5, 7], verbose=False)

    # Check if YOLO successfully locked onto objects in this frame
    if results[0].boxes.id is not None:
        boxes = results[0].boxes.xywh.cpu().numpy()
         track_ids = results[0].boxes.id.int().cpu().tolist()

        for box, track_id in zip(boxes, track_ids):
            x, y, w, h = box
            current_center = (float(x), float(y))

            # Record position history to calculate movement direction
            track_history[track_id].append(current_center)
            if len(track_history[track_id]) > FRAME_WINDOW:
                track_history[track_id].pop(0)

            # Physics engine calculations require at least 2 historical frames
            if len(track_history[track_id]) >= 2:
                old_y = track_history[track_id][0][1]
                new_y = track_history[track_id][-1][1]

                # Calculate pixel displacement per frame over our window
                pixel_displacement = abs(new_y - old_y)
                velocity_per_frame = pixel_displacement / len(track_history[track_id])

                # Determine direction relative to the screen coordinates (0,0 is top-left)
                if new_y > old_y:
                    direction = "APPROACHING"
                    box_color = (0, 0, 255)  # Alert Red
                else:
                    direction = "RECEDING"
                    box_color = (255, 0, 0)  # Safe Blue

                # --- LIVE IMPACT COUNTDOWN LOGIC ---
                # Calculate time-to-collision ONLY for vehicles above the target traveling downward
                if new_y < target_y_line and direction == "APPROACHING":
                    pixel_distance = target_y_line - new_y

                    # --- LIVE IMPACT COUNTDOWN LOGIC ---
                # Calculate time-to-collision ONLY for vehicles above the target traveling downward
                if new_y < target_y_line and direction == "APPROACHING":
                    pixel_distance = target_y_line - new_y

                    if velocity_per_frame > 0.1:  # Prevent division by zero
                        frames_to_impact = pixel_distance / velocity_per_frame
                        seconds_to_impact = frames_to_impact / ASSUMED_FPS
                        ticker_label = f"ID:{track_id} | IMPACT: {seconds_to_impact:.1f}s"
                    else:
                        ticker_label = f"ID:{track_id} | IDLE"
                elif direction == "RECEDING":
                    ticker_label = f"ID:{track_id} | CLEAR"
                else:
                    ticker_label = f"ID:{track_id} | PASSED"

                # Render object graphics bounding box on screen
                top_left = (int(x - w/2), int(y - h/2))
                bottom_right = (int(x + w/2), int(y + h/2))

                cv2.rectangle(frame, top_left, bottom_right, box_color, 2)
                cv2.putText(frame, ticker_label, (top_left[0], top_left[1] - 8),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, box_color, 2)

    # 3. Render the output window display frame
    cv2.imshow("NYSDOT Blind Curve Tracking Sandbox", frame)

    # Check for 'q' key to quit program smoothly
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# Clean up environment bindings when loop collapses
print("Closing demo window safely...")
cap.release()
cv2.destroyAllWindows()
cv2.waitKey(1)

In [ ]:
import streamlit as st
import cv2
import pandas as pd
from elasticsearch import Elasticsearch
from elasticsearch.helpers import bulk
import time

# --- STREAMLIT UI LAYOUT ---
st.set_page_config(page_title="I-87 Traffic Monitor", layout="wide")
st.title("🚗 Live I-87 Blind Curve Traffic & Impact Tracker")
st.write("This dashboard triggers live vehicle telemetry and pushes analytics to Elastic Cloud.")

# Sidebar Configuration Controls
st.sidebar.header("Deployment Configuration")
cloud_id_input = st.sidebar.text_input("Elastic Cloud ID", value="HackathonTrafficMonitor:dXMtY2VudHJhbDEuZ2NwLmNsb3VkLmVzLmlvOjQ0MyQyYzRiMmJjMDg4ZDg0OWRjODlhZmFjMWFlNzRkMGNhZSRhOTk1ZmE0MWJiOTQ0MmMwYmNjYzE0ZDYwMmNhYzM1NA==")
password_input = st.sidebar.text_input("Elastic Password", type="password", value="xeStqhvi3rK5WyCrocMmr8TE")

run_duration = st.sidebar.slider("Tracking Run Duration (Seconds)", min_value=10, max_value=300, value=30)
start_button = st.sidebar.button("🚀 Start Live Stream Processing")

# Main Page Containers
video_placeholder = st.empty()
status_placeholder = st.empty()

# --- PROCESSING LOOP ---
if start_button:
    status_placeholder.info("Initializing YOLO Model and connecting to NYSDOT Feed...")

    # Delayed imports so app loads instantly
    from ultralytics import YOLO
    from datetime import datetime
    from collections import defaultdict

    model = YOLO("yolov8n.pt")
    video_source = "https://s9.nysdot.skyvdn.com:443/rtplive/R1_059/playlist.m3u8"
    cap = cv2.VideoCapture(video_source)

    track_history = defaultdict(lambda: [])
    payloads_to_send = []

    start_time = time.time()

    while cap.isOpened():
        if (time.time() - start_time) >= run_duration:
            break

        success, frame = cap.read()
        if not success: break

        h_frame, w_frame, _ = frame.shape
        target_y_line = int(h_frame * 0.65)

        # Draw target line on frame
        cv2.line(frame, (0, target_y_line), (w_frame, target_y_line), (0, 165, 255), 3)

        # YOLO Processing
        results = model.track(frame, persist=True, classes=[2, 3, 5, 7], verbose=False)

        if results[0].boxes.id is not None:
            boxes = results[0].boxes.xywh.cpu().numpy()
            track_ids = results[0].boxes.id.int().cpu().tolist()

            for box, track_id in zip(boxes, track_ids):
                x, y, w, h = box
                current_time = datetime.utcnow()
                track_history[track_id].append((float(x), float(y)))

                if len(track_history[track_id]) >= 2:
                    old_y = track_history[track_id][0][1]
                    new_y = track_history[track_id][-1][1]

                    moving_toward = 1 if new_y > old_y else 0
                    moving_away = 0 if new_y > old_y else 1

                    # Generate live JSON packet sample
                    payloads_to_send.append({
                        "_index": "traffic-live-stream",
                        "_source": {
                            "timestamp": current_time.isoformat(),
                            "OBJECT_ID": track_id,
                            "MOVING_AWAY": moving_away,
                            "MOVING_TOWARD": moving_toward,
                            "VELOCITY": round(abs(new_y - old_y) * 4, 2),
                            "ACCELERATION": 0.0,
                            "MAX_IMPACT_SCORE": 0.0,
                            "COUNT_DOWN_RATE": 1.0
                        }
                    })

                    # Annotate frame
                    cv2.rectangle(frame, (int(x-w/2), int(y-h/2)), (int(x+w/2), int(y+h/2)), (0, 255, 0), 2)

        # Stream the live video frame directly into the browser webpage!
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        video_placeholder.image(frame_rgb, channels="RGB", use_container_width=True)

    cap.release()

    # Sync with Elastic
    status_placeholder.warning("Syncing captured metrics with Elastic Cloud...")
    try:
        es = Elasticsearch(cloud_id=cloud_id_input, basic_auth=("elastic", password_input))
        success, _ = bulk(es, payloads_to_send)
        status_placeholder.success(f"🎉 Session complete! Successfully uploaded {success} tracking frames directly to Elastic.")
    except Exception as e:
        status_placeholder.error(f"Elastic Sync Failed: {e}")

**REQUIREMENTS**
streamlit
opencv-python-headless
ultralytics
elasticsearch
pandas

**DOCKER FILE**

FROM python:3.10-slim

WORKDIR /app

RUN apt-get update && apt-get install -y libglib2.0-0 libgl1-mesa-glx && rm -rf /var/lib/apt/lists/*

COPY . .

RUN pip install --no-cache-dir -r requirements.txt

EXPOSE 8080

CMD ["streamlit", "run", "app.py", "--server.port=8080", "--server.address=0.0.0.0"]